# 🏥 Diyabet KDS - Model Eğitimi (Polars + PyTorch)
**Memory-Efficient LSTM Training**

Bu notebook:
- Polars ile veri okur (hızlı, düşük RAM)
- Numpy ile sequence oluşturur
- PyTorch ile GPU'da eğitir

## 1. Kurulum ve GPU

In [ ]:
!pip install polars -q

import torch
import polars as pl

print(f'PyTorch: {torch.__version__}')
print(f'Polars: {pl.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pickle
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.notebook import tqdm

WORK_DIR = '/content/drive/MyDrive/DiabetesKDS'
DATA_DIR = f'{WORK_DIR}/data/processed'
MODEL_DIR = f'{WORK_DIR}/models'
os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Config

In [ ]:
CONFIG = {
    # Columns
    'cgm_col': 'CGM',
    'bolus_col': 'bolus',
    'carb_col': 'carbs',
    'patient_col': 'id',
    'feature_cols': ['CGM', 'carbs', 'cgm_delta_5min', 'time_sin', 'time_cos'],
    
    # Sequence
    'seq_length': 12,       # 1 saat
    'pred_horizon': 6,      # 30 dakika
    
    # Training
    'batch_size': 512,
    'epochs': 30,
    'learning_rate': 0.001,
    'train_split': 0.8,
    'patience': 7,
    
    # Model
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.2,
    
    # Sampling (RAM için)
    'max_patients': None,    # None = hepsi
    'sample_fraction': 0.3   # %30 sequence
}

## 3. Veri Yükleme (Polars)

In [ ]:
%%time

print('Veri yükleniyor (Polars)...')

df = pl.read_parquet(f'{DATA_DIR}/metabonet_processed.parquet')

print(f'✅ Yüklendi: {len(df):,} satır')
print(f'   Hasta: {df[CONFIG["patient_col"]].n_unique():,}')
print(f'   RAM: {df.estimated_size("gb"):.2f} GB')
print(f'   Sütunlar: {df.columns}')

In [ ]:
df.head(5)

## 4. Sequence Oluşturma (Memory-Efficient)

In [ ]:
class PolarsSequenceDataset(Dataset):
    """Memory-efficient sequence dataset using Polars."""
    
    def __init__(self, df: pl.DataFrame, seq_length=12, pred_horizon=6,
                 max_patients=None, sample_fraction=1.0):
        self.seq_length = seq_length
        self.pred_horizon = pred_horizon
        
        feature_cols = CONFIG['feature_cols']
        target_col = CONFIG['bolus_col']
        patient_col = CONFIG['patient_col']
        
        # Hasta listesi
        patients = df[patient_col].unique().to_list()
        if max_patients:
            patients = patients[:max_patients]
        
        print(f'İşlenecek hasta: {len(patients)}')
        
        sequences = []
        targets = []
        
        for i, pid in enumerate(tqdm(patients, desc='Hasta')):
            # Polars filtering (çok hızlı)
            patient_df = df.filter(pl.col(patient_col) == pid)
            
            if len(patient_df) < seq_length + pred_horizon:
                continue
            
            # Numpy'a çevir (sadece gerekli sütunlar)
            features = patient_df.select(feature_cols).to_numpy().astype(np.float32)
            bolus = patient_df[target_col].to_numpy().astype(np.float32)
            
            # NaN handle
            features = np.nan_to_num(features, nan=0.0)
            bolus = np.nan_to_num(bolus, nan=0.0)
            
            # Sliding window
            n_sequences = len(features) - seq_length - pred_horizon + 1
            
            for j in range(n_sequences):
                # Sampling
                if sample_fraction < 1.0 and np.random.random() > sample_fraction:
                    continue
                
                seq = features[j:j + seq_length]
                target = bolus[j + seq_length:j + seq_length + pred_horizon].sum()
                
                sequences.append(seq)
                targets.append(target)
        
        self.sequences = np.array(sequences, dtype=np.float32)
        self.targets = np.array(targets, dtype=np.float32)
        
        print(f'\n✅ {len(self.sequences):,} sequence oluşturuldu')
        print(f'   Shape: X={self.sequences.shape}, y={self.targets.shape}')
        print(f'   RAM: {(self.sequences.nbytes + self.targets.nbytes) / 1e9:.2f} GB')
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return (torch.FloatTensor(self.sequences[idx]),
                torch.FloatTensor([self.targets[idx]]))
    
    def get_feature_dim(self):
        return self.sequences.shape[2]

In [ ]:
%%time

print('Dataset oluşturuluyor...')

dataset = PolarsSequenceDataset(
    df,
    seq_length=CONFIG['seq_length'],
    pred_horizon=CONFIG['pred_horizon'],
    max_patients=CONFIG['max_patients'],
    sample_fraction=CONFIG['sample_fraction']
)

# RAM'i boşalt - artık df'e gerek yok
del df
import gc
gc.collect()

In [ ]:
# Train/Val split
train_size = int(CONFIG['train_split'] * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f'Train: {len(train_dataset):,}')
print(f'Val: {len(val_dataset):,}')

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], 
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], 
                        shuffle=False, num_workers=2, pin_memory=True)

## 5. Model

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])


# Model oluştur
feature_dim = dataset.get_feature_dim()
model = LSTMPredictor(
    input_dim=feature_dim,
    hidden_dim=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout']
).to(DEVICE)

print(f'Model: {sum(p.numel() for p in model.parameters()):,} parametre')
print(model)

## 6. Eğitim

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    preds, targets = [], []
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            total_loss += criterion(out, y).item()
            preds.extend(out.cpu().numpy().flatten())
            targets.extend(y.cpu().numpy().flatten())
    
    preds, targets = np.array(preds), np.array(targets)
    mae = mean_absolute_error(targets, preds)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    
    return total_loss / len(loader), mae, rmse, preds, targets

In [ ]:
# Training
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_loss = float('inf')
best_state = None
no_improve = 0
history = {'train': [], 'val': [], 'mae': [], 'rmse': []}

print('='*60)
print('EĞİTİM')
print('='*60)

for epoch in range(CONFIG['epochs']):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, mae, rmse, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['mae'].append(mae)
    history['rmse'].append(rmse)
    
    print(f"Epoch {epoch+1:2d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | MAE: {mae:.3f}")
    
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = model.state_dict().copy()
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= CONFIG['patience']:
            print(f'\n⚠️ Early stopping')
            break

model.load_state_dict(best_state)
print('\n✅ En iyi model yüklendi')

## 7. Sonuçlar

In [ ]:
# Plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(history['train'], label='Train')
axes[0].plot(history['val'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['mae'], 'g-')
axes[1].set_title('MAE')
axes[1].grid(True)

axes[2].plot(history['rmse'], 'r-')
axes[2].set_title('RMSE')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_history.png', dpi=150)
plt.show()

In [ ]:
# Final metrics
_, mae, rmse, y_pred, y_true = evaluate(model, val_loader, criterion)

within_1 = np.mean(np.abs(y_pred - y_true) <= 1.0) * 100
within_2 = np.mean(np.abs(y_pred - y_true) <= 2.0) * 100

print('='*60)
print('FİNAL SONUÇLAR')
print('='*60)
print(f'MAE: {mae:.3f} units')
print(f'RMSE: {rmse:.3f} units')
print(f'±1 unit: {within_1:.1f}%')
print(f'±2 unit: {within_2:.1f}%')

In [ ]:
# Predictions plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_true, y_pred, alpha=0.2, s=3)
axes[0].plot([0, max(y_true.max(), y_pred.max())], [0, max(y_true.max(), y_pred.max())], 'r--')
axes[0].set_xlabel('Gerçek')
axes[0].set_ylabel('Tahmin')
axes[0].set_title('Gerçek vs Tahmin')
axes[0].grid(True)

axes[1].hist(y_pred - y_true, bins=50, alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_title('Hata Dağılımı')
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/predictions.png', dpi=150)
plt.show()

## 8. Kaydet

In [ ]:
model_path = f'{MODEL_DIR}/lstm_insulin_model.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'history': history,
    'feature_dim': feature_dim,
    'metrics': {'mae': mae, 'rmse': rmse, 'within_1': within_1, 'within_2': within_2}
}, model_path)

print(f'✅ Model: {model_path}')
print('\n🎉 TAMAMLANDI!')

In [ ]:
# İndir
from google.colab import files
files.download(model_path)